<a href="https://colab.research.google.com/github/mvashi-sonic/AICapstoneProj/blob/dev/capstone_FAISS_retriever.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install sentence-transformers faiss-cpu

Mount The Drive

In [2]:
from google.colab import drive

# 1. Mount your Google Drive
drive.mount('/content/drive')

Mounted at /content/drive


Import Required Packages

In [3]:
import json
import pickle
import numpy as np
import faiss

from sentence_transformers import SentenceTransformer

Sentence Tranformer

In [ ]:
embedder = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)
print(embedder.get_sentence_embedding_dimension())

Read the Data

In [ ]:
chunks = []

with open("/content/drive/MyDrive/capstone_usable_qa_data/all_records.json", encoding="utf-8") as f:
    records = json.load(f)
    for record in records["All_Records"]:
        chunks.append(record)


Create Pragraphs With Metadata

In [ ]:
paragraphs = [
    chunk["metadata"] + chunk["reference"]
    for chunk in chunks
]
print(paragraphs[0][:300])
print(paragraphs[1][:300])
print(paragraphs[2][:300])

Create Embeddings

In [ ]:
embeddings = embedder.encode(
    paragraphs,
    batch_size=128,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)
normalize_embeddings=True
print(embeddings.shape)

In [40]:

dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)


Write Indexes

In [ ]:
print(index.ntotal)
print(len(chunks))
print(len(embeddings))

faiss.write_index(
    index,
    "financial_reports.index"
)

In [42]:
metadata = []

for chunk in chunks:
    #print(f"{chunk["metadata"]}")
    tks = chunk["metadata"].split(" ")
    ticker = (tks[1]).split("-")[1]
    section = tks[len(tks)-1].split("-")[1]
    date = tks[len(tks)-2].split("-")[1]
    company = tks[2].split("-")[1]
    metadata.append({
        "ticker": ticker,
        "section": section,
        "year": date,
        "reference": chunk["reference"]
    })

In [43]:
with open("financial_reports_metadata.pkl", "wb") as f:
    pickle.dump(metadata, f)

Read Indexes

In [49]:
index = faiss.read_index(
    "financial_reports.index"
)
with open("financial_reports_metadata.pkl","rb") as f:
    metadata = pickle.load(f)
index = faiss.IndexFlatIP(384)
index.add(embeddings)

Search For Top 50 Answers Based On Question Encoding

In [ ]:
question = "What supplier risks does NVIDIA face in 2025?"

query_embedding = embedder.encode(
    [question],
    convert_to_numpy=True#,
    #normalize_embeddings=True
)

print(query_embedding.shape)
print(query_embedding[0][:10])

In [ ]:
scores, ids = index.search(
    query_embedding,
    k=50
)
print(scores)
print(ids)

In [ ]:
for score, idx in zip(scores[0], ids[0]):
    chunk = metadata[idx]

    file_output = open("retrieved_results.jsonl", "a")
    dict = {}
    dict["score"] = f"{score:.4f}"
    dict["ticker"] = chunk["ticker"]
    dict["section"] = chunk["section"]
    dict["year"] = chunk["section"]
    dict["reference"] = chunk["reference"]
    json.dump(dict, file_output)





Copy Results To Drive

In [ ]:
#keep
#Copy data to drive
import os
import shutil
from google.colab import drive

# 2. Define source and destination folders
# Replace 'my_folder' with the exact folder path where your .json files currently are
source_dir = '.'
# Replace 'My Drive/TargetFolder' with the Drive folder you want to copy to
destination_dir = '/content/drive/MyDrive/capstone_usable_qa_data'

# Create the destination directory if it doesn't exist
os.makedirs(destination_dir, exist_ok=True)

# 3. Find and copy all .json files
for filename in os.listdir(source_dir):
    if filename.endswith('retrieved_results.json'):
        source_file = os.path.join(source_dir, filename)
        destination_file = os.path.join(destination_dir, filename)

        shutil.copy2(source_file, destination_file)
        print(f"Copied: {filename}")

print("All .json files copied successfully!")